In [31]:
# install karo sab packages — pehle ye run karo
%pip install -r requirements.txt -q


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import re
import json
import time
import logging
from typing import TypedDict, List, Dict, Any, Optional, Literal
from dotenv import load_dotenv

# langchain wale imports
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# langgraph — the pitch where everything happens
from langgraph.graph import StateGraph, END

# pinecone for vector db — like a library but for AI
from pinecone import Pinecone

# env load karo
load_dotenv()

# logging setup — basic sa
logging.basicConfig(level=logging.INFO)
messi_logger = logging.getLogger("PACSA")

# ---- CONFIG ----
# tried using a dataclass first but dict is simpler lol
# from dataclasses import dataclass
# @dataclass
# class Config:
#     nvidia_key: str
#     pinecone_key: str

RONALDO_CONFIG = {
    "nvidia_api_key": os.getenv("NVIDIA_API_KEY"),
    "nvidia_model": os.getenv("NVIDIA_MODEL", "z-ai/glm5"),
    "pinecone_api_key": os.getenv("PINECONE_API_KEY"),
    "pinecone_index": os.getenv("PINECONE_INDEX", "adsparkx-assignment"),
    "pinecone_host": os.getenv("PINECONE_HOST"),
    "embed_model": os.getenv("PINECONE_EMBED_MODEL", "llama-text-embed-v2"),
    "embed_dim": 1024,
    "max_retries": 2,
    "quality_threshold": 0.75,
    "max_tokens": 16384,
    "temperature": 0.7,
}

goalkeeper = ChatNVIDIA(
    model=RONALDO_CONFIG["nvidia_model"],
    api_key=RONALDO_CONFIG["nvidia_api_key"],
    temperature=RONALDO_CONFIG["temperature"],
    max_tokens=RONALDO_CONFIG["max_tokens"],
    extra_body={"chat_template_kwargs": {"enable_thinking": False, "clear_thinking": False}},
    timeout=600, # 10 min timeout
)


pc = Pinecone(api_key=RONALDO_CONFIG["pinecone_api_key"])
pitch_index = pc.Index(
    RONALDO_CONFIG["pinecone_index"],
    host=RONALDO_CONFIG["pinecone_host"]
)

print("Config loaded | LLM ready | Pinecone connected")
print(f"Model: {RONALDO_CONFIG['nvidia_model']}")
print(f"Index: {RONALDO_CONFIG['pinecone_index']}")

/tmp/ipykernel_327367/1357124491.py:48: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  goalkeeper = ChatNVIDIA(


Config loaded | LLM ready | Pinecone connected
Model: z-ai/glm5
Index: adsparkx-assignment


In [33]:
# ---- AGENT STATE ----
# pehle pydantic use kiya tha but wo bahut complex hai for this
# from pydantic import BaseModel, Field
# class AgentState(BaseModel):
#     current_user_message: str = Field(default="")


class AgentState(TypedDict, total=False):
    current_user_message: str
    saaf_message: str  # cleaned/sanitized message
    conversation_history: List[Dict[str, str]]
    detected_persona: str  # TECHNICAL_EXPERT | FRUSTRATED_USER | BUSINESS_EXECUTIVE | GENERAL_USER
    persona_confidence: float
    kb_results: List[Dict[str, Any]]
    kb_query: str
    jawab: str
    quality_score: float
    quality_feedback: str
    retry_count: int
    should_escalate: bool
    escalation_wajah: str
    escalation_priority: str
    escalation_department: str
    escalation_summary: str
    is_resolved: bool
    route_decision: str
    bhasha: str

def make_fresh_state(user_msg: str) -> dict:
    # naya state banao for a new conversation
    return {
        "current_user_message": user_msg,
        "saaf_message": "",
        "conversation_history": [],
        "detected_persona": "GENERAL_USER",
        "persona_confidence": 0.0,
        "kb_results": [],
        "kb_query": "",
        "jawab": "",
        "quality_score": 0.0,
        "quality_feedback": "",
        "retry_count": 0,
        "should_escalate": False,
        "escalation_wajah": "",
        "escalation_priority": "medium",
        "escalation_department": "general_support",
        "escalation_summary": "",
        "is_resolved": False,
        "route_decision": "",
        "bhasha": "en",
    }

CHESS_PERSONAS = {
    "TECHNICAL_EXPERT": {
        "tone": "precise, code-heavy, no fluff",
        "kb_tier": "technical_deep",
        "max_tokens": 2048,
        "temp": 0.3,  # low temperature = more precise
    },
    "FRUSTRATED_USER": {
        "tone": "empathy-first, simple steps, offer escalation",
        "kb_tier": "simplified",
        "max_tokens": 1024,
        "temp": 0.7,
    },
    "BUSINESS_EXECUTIVE": {
        "tone": "concise metrics, executive summary, bold KPIs",
        "kb_tier": "executive_summary",
        "max_tokens": 800,
        "temp": 0.5,
    },
    "GENERAL_USER": {
        "tone": "warm, patient, step-by-step",
        "kb_tier": "simplified",
        "max_tokens": 1200,
        "temp": 0.7,
    },
}

print("State model and persona profiles ready")


State model and persona profiles ready


In [34]:
def saaf_karo(text: str) -> str:
    """sanitize input — remove html, extra spaces"""
    if not text or not isinstance(text, str):
        return ""
    saaf_text = re.sub(r'<[^>]+>', '', text)
    saaf_text = re.sub(r'\s+', ' ', saaf_text).strip()
    saaf_text = re.sub(r'<script.*?>.*?</script>', '', saaf_text, flags=re.DOTALL | re.IGNORECASE)
    return saaf_text


RED_CARD_WORDS = [
    "speak to human", "talk to agent", "real person",
    "supervisor", "manager", "lawyer", "sue",
    "legal action", "complaint", "escalate",
    "cancel my account", "close my account",
]

SIMPLE_INTENTS = {
    "greeting": ["hello", "hi", "hey", "good morning", "good afternoon", "good evening"],
    "farewell": ["bye", "goodbye", "see you", "thanks bye", "that's all"],
    "gratitude": ["thank you", "thanks", "appreciate it", "helpful", "great help"],
}

def check_red_card(text: str) -> bool:
    """check if user wants to talk to a human — immediate red card"""
    text_lower = text.lower()
    for keyword in RED_CARD_WORDS:
        if keyword in text_lower:
            return True
    return False

def detect_simple_intent(text: str) -> Optional[str]:
    """check if its just a greeting/bye/thanks — no need for the full formation"""
    text_lower = text.lower().strip()
    for intent, keywords in SIMPLE_INTENTS.items():
        for kw in keywords:
            if kw in text_lower:
                return intent
    return None

def detect_bhasha(text: str) -> str:
    """basic language detection — just english for now"""
    if re.search(r'[\u0900-\u097F]', text):
        return "hi"
    return "en"

print("Utilities loaded — pitch is clean")

Utilities loaded — pitch is clean


In [35]:
PERSONA_DETECTION_PROMPT = """You are a customer support persona classifier.
Analyze the following customer message and classify them into ONE of these personas:
1. TECHNICAL_EXPERT — uses error codes, API references, SDK versions, technical jargon
2. FRUSTRATED_USER — uses ALL CAPS, excessive punctuation (!!!, ???), threats to cancel, angry tone
3. BUSINESS_EXECUTIVE — mentions SLA, ROI, compliance, board meetings, timelines, metrics
4. GENERAL_USER — simple how-to questions, polite tone, basic product usage

Also provide a confidence score between 0.0 and 1.0.

Respond in this exact JSON format:
{{"persona": "PERSONA_NAME", "confidence": 0.85, "reasoning": "brief explanation"}}

Customer message: {message}"""

# the response generation prompts — different for each persona, like different tactics

RESPONSE_PROMPTS = {
    "TECHNICAL_EXPERT": """You are a senior technical support engineer. The customer is a technical expert.
Rules:
- Be precise and direct, skip unnecessary pleasantries
- Include code blocks, CLI commands, or API examples when relevant
- Structure: Root Cause → Solution → Verification Steps
- NEVER say "I understand your frustration" — they don't want empathy, they want solutions
- Reference documentation and version numbers

Context from knowledge base:
{kb_context}

Previous conversation:
{history}

Customer query: {message}

Provide a technical, code-heavy response.""",

    "FRUSTRATED_USER": """You are a caring customer support agent. The customer is frustrated.
Rules:
- Your FIRST sentence MUST be a specific empathy statement (use words like: sorry, hear, understand, apologize)
- Keep it simple — maximum 3 action steps
- No technical jargon
- Always offer escalation to a human agent as a fallback
- Be patient and acknowledge their frustration specifically

Context from knowledge base:
{kb_context}

Previous conversation:
{history}

Customer query: {message}

Provide an empathetic, simple response.""",

    "BUSINESS_EXECUTIVE": """You are an executive account manager. The customer is a business executive.
Rules:
- Lead with business impact and metrics
- Bold all numbers, percentages, SLAs using markdown **bold**
- Use bullet points, keep total response under 300 tokens
- Focus on timelines, ROI, and compliance
- Be concise — executives skim, they don't read

Context from knowledge base:
{kb_context}

Previous conversation:
{history}

Customer query: {message}

Provide a concise, metrics-driven executive summary.""",

    "GENERAL_USER": """You are a friendly customer support agent. The customer is a general user.
Rules:
- Be warm and patient
- Explain step-by-step with visual cues ("click the blue button", "look for the gear icon")
- No jargon — explain things simply
- Anticipate the next logical question and address it
- End with "Is there anything else I can help with?"

Context from knowledge base:
{kb_context}

Previous conversation:
{history}

Customer query: {message}

Provide a warm, step-by-step response.""",
}

# quality gate prompt — the VAR check, like in football
QUALITY_GATE_PROMPT = """You are a quality evaluator for customer support responses.
Evaluate the following response on these 4 dimensions (score each 0.0 to 1.0):

1. HALLUCINATION (weight: 0.35) — Is the response grounded in the provided KB context? Penalize made-up info.
2. TONE_ALIGNMENT (weight: 0.25) — Does the tone match the {persona} persona? 
3. COMPLETENESS (weight: 0.30) — Does it fully address the customer's query with actionable steps?
4. SAFETY (weight: 0.10) — Is it free of PII leaks, unauthorized promises, or harmful content?

Customer persona: {persona}
Customer query: {query}
KB context provided: {kb_context}
Generated response: {response}

Respond in this exact JSON format:
{{"hallucination_score": 0.8, "tone_score": 0.9, "completeness_score": 0.7, "safety_score": 1.0, "overall_score": 0.82, "feedback": "specific improvement suggestions"}}"""

# escalation summary prompt
ESCALATION_SUMMARY_PROMPT = """Summarize this customer interaction for a human agent handoff.
Include:
1. Customer's core issue
2. Detected persona and emotional state
3. What was attempted by the AI
4. Recommended priority (low/medium/high/critical)
5. Suggested department (billing/technical/security/general_support)

Conversation history: {history}
Current message: {message}
Detected persona: {persona}
Escalation reason: {reason}

Provide a concise handoff summary."""

# tried using langchain PromptTemplate but string format is simpler
# from langchain.prompts import PromptTemplate
# persona_prompt = PromptTemplate(input_variables=["message"], template=PERSONA_DETECTION_PROMPT)
# ^ ye bhi kaam karta but .format() is easier to understand

print("✅ Prompt templates loaded — playbook ready")


✅ Prompt templates loaded — playbook ready


In [36]:
# ---- KNOWLEDGE BASE ----
# the library of our support system — like a team's video analysis room

# sample KB data — in real life this would come from a database
SAMPLE_KB_DATA = [
    {
        "id": "kb_001",
        "text": "To reset your password: Go to Settings > Security > Change Password. Enter your current password, then your new password twice. Click Save. If you forgot your password, click 'Forgot Password' on the login page and check your email.",
        "content_tier": "simplified",
        "category": "account",
    },
    {
        "id": "kb_002",
        "text": "API Rate Limiting: Our API enforces rate limits of 1000 requests/minute for Pro plans and 100 requests/minute for Free plans. When rate limited, you'll receive HTTP 429 with a Retry-After header. Implement exponential backoff: start with 1s delay, double on each retry, max 32s. Example: curl -H 'Authorization: Bearer TOKEN' https://api.example.com/v2/data",
        "content_tier": "technical_deep",
        "category": "api",
    },
    {
        "id": "kb_003",
        "text": "Billing FAQ: We offer monthly and annual plans. Annual plans save 20%. To upgrade: Dashboard > Billing > Change Plan. Refunds are available within 14 days of purchase. For enterprise pricing, contact sales@example.com.",
        "content_tier": "simplified",
        "category": "billing",
    },
    {
        "id": "kb_004",
        "text": "Platform SLA: We guarantee 99.9% uptime for Enterprise plans. Current month uptime: 99.95%. Mean response time: 145ms (P95: 320ms). Incident response: P1 within 15min, P2 within 1hr. SLA credits: 10% for each 0.1% below target. Compliance: SOC2 Type II, GDPR, HIPAA ready.",
        "content_tier": "executive_summary",
        "category": "platform",
    },
    {
        "id": "kb_005",
        "text": "OAuth2 PKCE Flow Implementation: 1) Generate code_verifier (43-128 chars, [A-Za-z0-9-._~]). 2) Create code_challenge = BASE64URL(SHA256(code_verifier)). 3) Authorization request: GET /authorize?response_type=code&client_id=CLIENT&code_challenge=CHALLENGE&code_challenge_method=S256. 4) Exchange code: POST /token with code, code_verifier, grant_type=authorization_code. Common error: invalid_grant usually means code_verifier mismatch or expired auth code (5min TTL).",
        "content_tier": "technical_deep",
        "category": "authentication",
    },
    {
        "id": "kb_006",
        "text": "Getting Started Guide: Welcome! To set up your account: 1) Click 'Sign Up' on our homepage. 2) Enter your email and create a password. 3) Verify your email by clicking the link we send. 4) Complete your profile. 5) You're ready to go! Need help? Live chat is available 9am-5pm EST.",
        "content_tier": "simplified",
        "category": "onboarding",
    },
    {
        "id": "kb_007",
        "text": "Webhook Configuration: Set up webhooks at Dashboard > Integrations > Webhooks. Supported events: user.created, payment.completed, subscription.cancelled. Payload format: JSON with HMAC-SHA256 signature in X-Signature header. Verify: hash = HMAC(payload, webhook_secret). Retry policy: 3 attempts with exponential backoff. Debug: Check webhook logs at /dashboard/webhooks/logs.",
        "content_tier": "technical_deep",
        "category": "integrations",
    },
    {
        "id": "kb_008",
        "text": "Enterprise ROI Report: Average customer sees 40% reduction in support ticket volume after implementing our AI assistant. Mean time to resolution decreased by 62%. Customer satisfaction scores improved from 3.2 to 4.6 out of 5. Annual cost savings: $150,000-$500,000 depending on team size. Implementation timeline: 2-4 weeks.",
        "content_tier": "executive_summary",
        "category": "enterprise",
    },
    {
        "id": "kb_009",
        "text": "Security Incident Response: If you suspect unauthorized access: 1) Immediately change your password. 2) Enable 2FA at Settings > Security > Two-Factor Auth. 3) Review active sessions at Settings > Security > Active Sessions and revoke unknown ones. 4) Contact security@example.com. 5) We will investigate within 24 hours and provide a detailed report.",
        "content_tier": "simplified",
        "category": "security",
    },
    {
        "id": "kb_010",
        "text": "Refund Policy: Full refund available within 14 days of purchase, no questions asked. After 14 days, prorated refund for annual plans. Monthly plans: no refund after billing cycle starts. Process: Dashboard > Billing > Request Refund, or email billing@example.com. Processing time: 5-10 business days.",
        "content_tier": "simplified",
        "category": "billing",
    },
]


def ingest_kb_to_pinecone():
    """KB data ko pinecone mein daal do — one time setup"""
    messi_logger.info("Starting KB ingestion to Pinecone...")

    try:
        stats = pitch_index.describe_index_stats()
        total_vectors = getattr(stats, 'total_vector_count', 0) if hasattr(stats, 'total_vector_count') else stats.get("total_vector_count", 0)

        if total_vectors > 0:
            messi_logger.info(f"Already {total_vectors} vectors in index, skipping ingestion")
            print(f"Index already has {total_vectors} vectors — skipping ingestion")
            return True

        texts_to_embed = [item["text"] for item in SAMPLE_KB_DATA]

        embeddings = pc.inference.embed(
            model=RONALDO_CONFIG["embed_model"],
            inputs=texts_to_embed,
            parameters={"input_type": "passage"}
        )

        vectors_to_upsert = []
        for i, item in enumerate(SAMPLE_KB_DATA):
            emb_values = embeddings[i].values if hasattr(embeddings[i], 'values') else embeddings[i]["values"]
            vectors_to_upsert.append({
                "id": item["id"],
                "values": emb_values,
                "metadata": {
                    "text": item["text"],
                    "content_tier": item["content_tier"],
                    "category": item["category"],
                }
            })

        pitch_index.upsert(vectors=vectors_to_upsert)

        time.sleep(2)
        messi_logger.info(f"Ingested {len(vectors_to_upsert)} vectors")
        print(f"Ingested {len(vectors_to_upsert)} KB docs into Pinecone")
        return True

    except Exception as e:
        messi_logger.error(f"Ingestion failed: {e}")
        print(f"KB ingestion failed: {e}")
        return False


def search_kb(query: str, content_tier: str = None, top_k: int = 3) -> List[Dict]:
    """KB mein search karo — 2 pass strategy like chess opening"""
    try:
        # embed the query
        query_embedding = pc.inference.embed(
            model=RONALDO_CONFIG["embed_model"],
            inputs=[query],
            parameters={"input_type": "query"}
        )

        qvec = query_embedding[0].values if hasattr(query_embedding[0], 'values') else query_embedding[0]["values"]

        results = []
        if content_tier:
            response = pitch_index.query(
                vector=qvec,
                top_k=top_k,
                include_metadata=True,
                filter={"content_tier": {"$eq": content_tier}}
            )
            results = response.matches if hasattr(response, 'matches') else response.get("matches", [])

        if not results:
            response = pitch_index.query(
                vector=qvec,
                top_k=top_k,
                include_metadata=True
            )
            results = response.matches if hasattr(response, 'matches') else response.get("matches", [])

        kb_docs = []
        for match in results:
            meta = match.metadata if hasattr(match, 'metadata') else match.get("metadata", {})
            score = match.score if hasattr(match, 'score') else match.get("score", 0)
            kb_docs.append({
                "text": meta.get("text", "") if isinstance(meta, dict) else getattr(meta, "text", ""),
                "score": score,
                "tier": meta.get("content_tier", "unknown") if isinstance(meta, dict) else getattr(meta, "content_tier", "unknown"),
                "category": meta.get("category", "unknown") if isinstance(meta, dict) else getattr(meta, "category", "unknown"),
            })

        return kb_docs

    except Exception as e:
        messi_logger.error(f"KB search failed: {e}")
        return []


# run ingestion
ingest_kb_to_pinecone()


INFO:PACSA:Starting KB ingestion to Pinecone...
INFO:PACSA:Already 57 vectors in index, skipping ingestion


Index already has 57 vectors — skipping ingestion


True

In [37]:
def intake_node(state: dict) -> dict:
    messi_logger.info("Intake node — cleaning the message")

    raw_msg = state.get("current_user_message", "")
    saaf = saaf_karo(raw_msg)
    bhasha = detect_bhasha(raw_msg)

    needs_red_card = check_red_card(saaf)

    simple = detect_simple_intent(saaf)

    updates = {
        "saaf_message": saaf,
        "bhasha": bhasha,
    }

    if needs_red_card:
        updates["should_escalate"] = True
        updates["escalation_wajah"] = "Customer explicitly requested human agent"
        updates["route_decision"] = "escalation"
    elif simple:
        updates["route_decision"] = f"simple_{simple}"
    else:
        updates["route_decision"] = "needs_analysis"

    return updates


def persona_detection_node(state: dict) -> dict:
    messi_logger.info("Persona detection — VAR reviewing the play")

    saaf = state.get("saaf_message", "")

    if state.get("should_escalate"):
        return {}

    if state.get("route_decision", "").startswith("simple_"):
        return {"detected_persona": "GENERAL_USER", "persona_confidence": 0.9}

    try:
        prompt = PERSONA_DETECTION_PROMPT.format(message=saaf)
        ai_msg = goalkeeper.invoke([HumanMessage(content=prompt)])
        response_text = ai_msg.content.strip()


        json_start = response_text.find('{')
        json_end = response_text.rfind('}') + 1
        if json_start >= 0 and json_end > json_start:
            json_str = response_text[json_start:json_end]
            parsed = json.loads(json_str)
        else:
            parsed = {"persona": "GENERAL_USER", "confidence": 0.5, "reasoning": "could not parse"}

        detected = parsed.get("persona", "GENERAL_USER")
        confidence = float(parsed.get("confidence", 0.5))

        valid_personas = ["TECHNICAL_EXPERT", "FRUSTRATED_USER", "BUSINESS_EXECUTIVE", "GENERAL_USER"]
        if detected not in valid_personas:
            detected = "GENERAL_USER"
            confidence = 0.5

        messi_logger.info(f"   Detected: {detected} (confidence: {confidence})")

        return {
            "detected_persona": detected,
            "persona_confidence": confidence,
        }

    except Exception as e:
        messi_logger.error(f"Persona detection failed: {e}")
        return {"detected_persona": "GENERAL_USER", "persona_confidence": 0.3}


def kb_retrieval_node(state: dict) -> dict:
    messi_logger.info("KB Retrieval — checking the tactical notebook")

    saaf = state.get("saaf_message", "")
    persona = state.get("detected_persona", "GENERAL_USER")

    tier = CHESS_PERSONAS.get(persona, {}).get("kb_tier", "simplified")

    try:
        reformulate_prompt = f"Rewrite this customer support query as a concise search query for a knowledge base. Only output the search query, nothing else.\n\nQuery: {saaf}"
        reformulated = goalkeeper.invoke([HumanMessage(content=reformulate_prompt)])
        search_query = reformulated.content.strip()
    except:
        search_query = saaf  # fallback to original

    results = search_kb(search_query, content_tier=tier, top_k=3)

    kuch_mila = len(results) > 0  # did we find anything?

    messi_logger.info(f"   Found {len(results)} results (tier: {tier})")

    return {
        "kb_results": results,
        "kb_query": search_query,
    }


def response_generation_node(state: dict) -> dict:
    messi_logger.info("Response generation — time to score!")

    saaf = state.get("saaf_message", "")
    persona = state.get("detected_persona", "GENERAL_USER")
    kb_results = state.get("kb_results", [])
    history = state.get("conversation_history", [])
    quality_feedback = state.get("quality_feedback", "")

    if kb_results:
        kb_context = "\n---\n".join([r["text"] for r in kb_results])
    else:
        kb_context = "No specific knowledge base articles found. Use general knowledge."

    if history:
        history_str = "\n".join([f"{h['role']}: {h['content']}" for h in history[-5:]])
    else:
        history_str = "No previous conversation."

    prompt_template = RESPONSE_PROMPTS.get(persona, RESPONSE_PROMPTS["GENERAL_USER"])
    prompt = prompt_template.format(
        kb_context=kb_context,
        history=history_str,
        message=saaf,
    )

    if quality_feedback:
        prompt += f"\n\nPREVIOUS ATTEMPT FEEDBACK (improve on this):\n{quality_feedback}"

    try:
        persona_config = CHESS_PERSONAS.get(persona, CHESS_PERSONAS["GENERAL_USER"])


        ai_msg = goalkeeper.invoke([
            SystemMessage(content=f"Respond with the tone: {persona_config['tone']}"),
            HumanMessage(content=prompt)
        ])

        jawab = ai_msg.content.strip()
        messi_logger.info(f"   Generated response ({len(jawab)} chars)")

        return {"jawab": jawab}

    except Exception as e:
        messi_logger.error(f"Response generation failed: {e}")
        return {
            "jawab": "I apologize, but I'm having trouble generating a response. Let me connect you with a team member who can help.",
            "should_escalate": True,
            "escalation_wajah": f"Response generation error: {str(e)}",
        }


def direct_response_node(state: dict) -> dict:
    messi_logger.info("Direct response — quick pass")

    route = state.get("route_decision", "")

    templates = {
        "simple_greeting": "Hello! Welcome to our support team. How can I help you today?",
        "simple_farewell": "Thank you for contacting us! Have a great day! If you need anything else, we're always here.",
        "simple_gratitude": "You're welcome! I'm glad I could help. Is there anything else you'd like to know?",
    }

    jawab = templates.get(route, "Hi there! How can I assist you?")

    return {"jawab": jawab, "quality_score": 0.95}  # pre-approved quality for templates


def quality_gate_node(state: dict) -> dict:
    messi_logger.info("Quality gate — referee reviewing the goal")

    jawab = state.get("jawab", "")
    persona = state.get("detected_persona", "GENERAL_USER")
    saaf = state.get("saaf_message", "")
    kb_results = state.get("kb_results", [])

    if state.get("quality_score", 0) >= 0.9:
        messi_logger.info("   Pre-approved quality — skipping check")
        return {}

    kb_context = "\n---\n".join([r["text"] for r in kb_results]) if kb_results else "No KB context"

    try:
        prompt = QUALITY_GATE_PROMPT.format(
            persona=persona,
            query=saaf,
            kb_context=kb_context,
            response=jawab,
        )

        ai_msg = goalkeeper.invoke([HumanMessage(content=prompt)])
        response_text = ai_msg.content.strip()

        json_start = response_text.find('{')
        json_end = response_text.rfind('}') + 1

        if json_start >= 0 and json_end > json_start:
            scores = json.loads(response_text[json_start:json_end])
        else:
            scores = {"overall_score": 0.8, "feedback": "Could not parse quality scores"}

        overall = float(scores.get("overall_score", 0.8))


        if persona == "FRUSTRATED_USER":
            pehla_line = jawab.split('.')[0].lower() if jawab else ""
            empathy_words = ["sorry", "apologize", "hear", "understand", "acknowledge"]
            has_empathy = any(w in pehla_line for w in empathy_words)
            if not has_empathy:
                tone_score = scores.get("tone_score", 0.7)
                scores["tone_score"] = tone_score * 0.5
                overall = (
                    float(scores.get("hallucination_score", 0.8)) * 0.35 +
                    float(scores.get("tone_score", 0.5)) * 0.25 +
                    float(scores.get("completeness_score", 0.7)) * 0.30 +
                    float(scores.get("safety_score", 0.9)) * 0.10
                )
                messi_logger.info("   Frustrated user penalty — no empathy detected")

        if persona == "TECHNICAL_EXPERT":
            faaltu_empathy = ["i understand your frustration", "i can see how frustrating"]
            for phrase in faaltu_empathy:
                if phrase in jawab.lower():
                    scores["tone_score"] = float(scores.get("tone_score", 0.7)) * 0.3
                    overall = (
                        float(scores.get("hallucination_score", 0.8)) * 0.35 +
                        float(scores.get("tone_score", 0.5)) * 0.25 +
                        float(scores.get("completeness_score", 0.7)) * 0.30 +
                        float(scores.get("safety_score", 0.9)) * 0.10
                    )
                    messi_logger.info("   Tech expert penalty — unnecessary empathy fluff")
                    break

        paas_hua = overall >= RONALDO_CONFIG["quality_threshold"]  # did it pass?
        retry = state.get("retry_count", 0)

        messi_logger.info(f"   Score: {overall:.2f} | Pass: {paas_hua} | Retry: {retry}")

        updates = {
            "quality_score": overall,
            "quality_feedback": scores.get("feedback", ""),
        }

        if not paas_hua:
            if retry < RONALDO_CONFIG["max_retries"]:
                updates["retry_count"] = retry + 1
            else:
                updates["should_escalate"] = True
                updates["escalation_wajah"] = f"Quality gate failed after {retry} retries (score: {overall:.2f})"

        return updates

    except Exception as e:
        messi_logger.error(f"Quality gate error: {e}")
        return {"quality_score": 0.7, "quality_feedback": f"Quality check error: {str(e)}"}


def escalation_node(state: dict) -> dict:
    messi_logger.info("Escalation — substituting in a human agent")

    saaf = state.get("saaf_message", "")
    persona = state.get("detected_persona", "GENERAL_USER")
    history = state.get("conversation_history", [])
    wajah = state.get("escalation_wajah", "Unknown reason")

    department = "general_support"
    priority = "medium"

    msg_lower = saaf.lower()
    if any(w in msg_lower for w in ["bill", "charge", "refund", "payment", "invoice"]):
        department = "billing"
    elif any(w in msg_lower for w in ["hack", "breach", "unauthorized", "security", "password"]):
        department = "security"
        priority = "high"
    elif any(w in msg_lower for w in ["api", "sdk", "error", "bug", "crash"]):
        department = "technical"

    if persona == "FRUSTRATED_USER":
        priority = "high"
    elif persona == "BUSINESS_EXECUTIVE":
        priority = "high"  # executives = VIP

    try:
        history_str = "\n".join([f"{h['role']}: {h['content']}" for h in history[-5:]]) if history else "No history"

        prompt = ESCALATION_SUMMARY_PROMPT.format(
            history=history_str,
            message=saaf,
            persona=persona,
            reason=wajah,
        )
        ai_msg = goalkeeper.invoke([HumanMessage(content=prompt)])
        summary = ai_msg.content.strip()
    except Exception as e:
        summary = f"Customer ({persona}) needs help. Issue: {saaf}. Reason for escalation: {wajah}"

    dept_display = department.replace('_', ' ').title()
    escalation_jawab = f"""**Transferring you to a human agent**

I understand you need additional assistance. I'm connecting you with our **{dept_display}** team.

**What happens next:**
- A support agent will be with you shortly
- Priority: **{priority.upper()}**
- Your conversation history has been shared for a seamless handoff

Thank you for your patience!"""

    messi_logger.info(f"   Department: {department} | Priority: {priority}")

    return {
        "jawab": escalation_jawab,
        "should_escalate": True,
        "escalation_department": department,
        "escalation_priority": priority,
        "escalation_summary": summary,
        "is_resolved": False,
    }


def output_node(state: dict) -> dict:
    messi_logger.info("Output node — final whistle!")

    jawab = state.get("jawab", "")
    saaf = state.get("saaf_message", "")

    history = list(state.get("conversation_history", []))
    history.append({"role": "user", "content": saaf})
    history.append({"role": "assistant", "content": jawab})

    return {
        "conversation_history": history,
        "is_resolved": True,
    }


print("All 8 nodes ready — full squad assembled")


All 8 nodes ready — full squad assembled


In [38]:
def route_after_persona(state: dict) -> str:
    """decide where to go after persona detection — chess opening"""
    # check if escalation needed
    if state.get("should_escalate"):
        return "escalation"

    route = state.get("route_decision", "needs_analysis")

    # simple intents go to direct response
    if route.startswith("simple_"):
        return "direct_response"

    # everything else goes to KB retrieval
    return "kb_retrieval"


def route_after_quality(state: dict) -> str:
    """decide after quality check — VAR decision"""
    # if escalation was triggered
    if state.get("should_escalate"):
        return "escalation"

    score = state.get("quality_score", 0)
    retry = state.get("retry_count", 0)

    # passed quality gate — GOAL!
    if score >= RONALDO_CONFIG["quality_threshold"]:
        return "output"

    # can still retry — VAR is still reviewing
    if retry <= RONALDO_CONFIG["max_retries"]:
        return "response_generation"

    # exhausted retries — red card, escalate
    return "escalation"


print("Routing edges defined — tactical plan is set")


Routing edges defined — tactical plan is set


In [39]:
formation = StateGraph(AgentState)
formation.add_node("intake", intake_node)
formation.add_node("persona_detection", persona_detection_node)
formation.add_node("kb_retrieval", kb_retrieval_node)
formation.add_node("response_generation", response_generation_node)
formation.add_node("direct_response", direct_response_node)
formation.add_node("quality_gate", quality_gate_node)
formation.add_node("escalation", escalation_node)
formation.add_node("output", output_node)
formation.set_entry_point("intake")
formation.add_edge("intake", "persona_detection")
formation.add_conditional_edges(
    "persona_detection",
    route_after_persona,
    {
        "escalation": "escalation",
        "direct_response": "direct_response",
        "kb_retrieval": "kb_retrieval",
    }
)

formation.add_edge("kb_retrieval", "response_generation")

formation.add_edge("response_generation", "quality_gate")

formation.add_edge("direct_response", "quality_gate")

formation.add_conditional_edges(
    "quality_gate",
    route_after_quality,
    {
        "output": "output",
        "response_generation": "response_generation",
        "escalation": "escalation",
    }
)

formation.add_edge("output", END)
formation.add_edge("escalation", END)

agent_graph = formation.compile()

print("Agent graph compiled — formation is ready!")
print("Nodes: intake -> persona -> [route] -> kb/direct/escalate -> quality -> [route] -> output/retry/escalate")


Agent graph compiled — formation is ready!
Nodes: intake -> persona -> [route] -> kb/direct/escalate -> quality -> [route] -> output/retry/escalate


In [40]:
def run_test(test_name: str, message: str):
    """run a test case — like playing a practice match"""
    # create fresh state
    initial_state = make_fresh_state(message)

    try:
        # run the graph
        result = agent_graph.invoke(initial_state)

        print(f"Persona: {result.get('detected_persona', 'N/A')} (confidence: {result.get('persona_confidence', 0):.2f})")
        print(f"Quality: {result.get('quality_score', 0):.2f}")
        print(f"Escalated: {result.get('should_escalate', False)}")

        if result.get("should_escalate"):
            print(f"   Department: {result.get('escalation_department', 'N/A')}")
            print(f"   Priority: {result.get('escalation_priority', 'N/A')}")
            print(f"   Reason: {result.get('escalation_wajah', 'N/A')}")

        print(f"\nResponse:")
        print(result.get("jawab", "No response generated"))
        print(f"{'='*60}")
        return result

    except Exception as e:
        print(f"Test failed: {e}")
        import traceback
        traceback.print_exc()
        return None


# ---- TEST CASES ----
# 5 practice matches to test our formation

print("\nRunning test suite...\n")

# Match 1: Technical Expert — the chess grandmaster
run_test(
    "Technical Expert Query",
    "I'm getting a 429 Too Many Requests error when calling the /v2/data endpoint with my OAuth2 PKCE flow. I've implemented exponential backoff but the Retry-After header shows inconsistent values. Using SDK v3.2.1. What's the correct implementation?"
)

# # Match 2: Frustrated User — needs empathy first
# run_test(
#     "Frustrated User",
#     "THIS IS RIDICULOUS!!! I've been charged TWICE for my subscription and nobody is helping me!! I want my money back RIGHT NOW or I'm cancelling everything!!!"
# )

# # Match 3: Business Executive — wants metrics and summary
# run_test(
#     "Business Executive",
#     "I need the latest SLA compliance report for our enterprise account. We have a board meeting next week and I need to present the ROI of this platform investment."
# )

# # Match 4: General User — needs patient help
# run_test(
#     "General User",
#     "Hi, how do I reset my password? I can't seem to find where to do it."
# )

# # Match 5: Escalation Request — immediate red card
# run_test(
#     "Escalation Request",
#     "I need to speak to a human agent right now. The AI is not understanding my issue and I want to talk to your supervisor."
# )

# print("\nAll test matches completed!")


INFO:PACSA:Intake node — cleaning the message
INFO:PACSA:Persona detection — VAR reviewing the play



Running test suite...



INFO:PACSA:   Detected: TECHNICAL_EXPERT (confidence: 0.98)
INFO:PACSA:KB Retrieval — checking the tactical notebook
INFO:PACSA:   Found 3 results (tier: technical_deep)
INFO:PACSA:Response generation — time to score!
INFO:PACSA:   Generated response (2770 chars)
INFO:PACSA:Quality gate — referee reviewing the goal
INFO:PACSA:   Score: 0.64 | Pass: False | Retry: 0
INFO:PACSA:Response generation — time to score!


KeyboardInterrupt: 